In [ ]:
import torch


print("CUDA Available:", torch.cuda.is_available())

print("PyTorch Version:", torch.__version__)


onnxruntime (CPU) and onnxruntime-gpu serve as highly optimized acceleration engines to run machine learning models that have been exported to the open-source Open Neural Network Exchange (ONNX) format. [1] (https://pypi.org/project/onnxruntime-gpu/), [2] (https://pypi.org/project/onnxruntime/)When combined with packages like transformers and torchaudio, they allow you to take deep learning models (like text, speech, or audio transformers) and run them much faster with lower memory usage than standard frameworks.

In [ ]:
import torch
import torchaudio

print("Torch version:", torch.__version__)
print("Audio version:", torchaudio.__version__)


In [ ]:
from transformers import AutoModel
import torch, torchaudio

# Load the model
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)

Doing a test with hindi sample.

In [ ]:
import soundfile as sf

In [ ]:
from transformers import AutoModel

# Audio path
audio_path = r"Audio Samples\Hindi Sample.wav"

# Load audio using soundfile instead of torchaudio
audio, sr = sf.read(audio_path)

# Convert numpy array to PyTorch tensor
wav = torch.tensor(audio, dtype=torch.float32)

# Convert stereo to mono
if wav.ndim > 1:
    wav = torch.mean(wav, dim=1)

# Add channel dimension
wav = wav.unsqueeze(0)

# Resample to 16kHz
target_sample_rate = 16000

if sr != target_sample_rate:

    resampler = torchaudio.transforms.Resample(
        orig_freq=sr,
        new_freq=target_sample_rate
    )

    wav = resampler(wav)


In [ ]:

# CTC Transcription
transcription_ctc = model(wav, "hi", "ctc")

print("CTC Transcription:")
print(transcription_ctc)



In [ ]:

# RNNT Transcription
transcription_rnnt = model(wav, "hi", "rnnt")

print("\nRNNT Transcription:")
print(transcription_rnnt)

RNNT-6.4 CTC-4.8 CTC is better for elders.

In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import soundfile as sf

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = ParlerTTSForConditionalGeneration.from_pretrained("ai4bharat/indic-parler-tts").to(device)
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts")
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)




In [ ]:
prompt = "Hey, how are you doing today?"
description = "A female speaker with a British accent delivers a slightly expressive and animated speech with a moderate speed and pitch. The recording is of very high quality, with the speaker's voice sounding clear and very close up."

In [5]:
import time
import os
import soundfile as sf
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

# 1. Organize samples into a list
samples = [
    {
        "language": "Assamese",
        "code": "as",
        "prompt": "নমস্কাৰ, আপুনি আজি কেনে আছে? বতৰটো আজি বৰ ধুনীয়া।",
        "description": "Sita speaks at a moderate pace with a clear, warm, and natural tone in a close-sounding recording environment with high audio quality and no background noise.",
    },
    {
        "language": "Bodo",
        "code": "brx",
        "prompt": "खुलुमबाय, नोंहा दिनै माबोरै दं? जोबोर गोजोननाय मोनबाय।",
        "description": "Bikram speaks with a clear, expressive voice at a normal pace and moderate pitch, delivered in high recording quality with very little ambient sound.",
    },
    {
        "language": "Nepali",
        "code": "ne",
        "prompt": "नमस्ते, तपाईंलाई आज कस्तो छ? आजको दिन धेरै राम्रो छ।",
        "description": "Amrita delivers a clear and slightly expressive speech at a calm, moderate pace and natural pitch, captured in a close environment with excellent audio quality.",
    },
    {
        "language": "Manipuri",
        "code": "mni",
        "prompt": "খুরুমজরি, অদোম ঙসি করি তৌরিগে? ঙসিগী নুমিৎসি য়াম্না ফৈ।",
        "description": "Laishram delivers clear and articulate speech at a steady, moderate pace and natural pitch in a high-quality studio environment with very clear audio.",
    },
]


In [6]:



def generate_and_save_tts(samples_list: list, output_dir: str = "tts_outputs"):
    """
    Runs Indic-Parler-TTS inference for each sample, benchmarks the time taken,
    and writes out the resulting audio files.
    """
    os.makedirs(output_dir, exist_ok=True)
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Loading models on device: {device}...")

    # Load model and tokenizers

    sampling_rate = model.config.sampling_rate

    print("\nStarting batch generation...\n" + "-" * 50)

    for item in samples_list:
        lang = item["language"]
        prompt = item["prompt"]
        desc = item["description"]

        # Prepare inputs
        desc_inputs = description_tokenizer(desc, return_tensors="pt").to(device)
        prompt_inputs = tokenizer(prompt, return_tensors="pt").to(device)

        # Synchronize CUDA before starting timer for accurate latency measurement
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.inference_mode():
            generation = model.generate(
                input_ids=desc_inputs.input_ids,
                attention_mask=desc_inputs.attention_mask,
                prompt_input_ids=prompt_inputs.input_ids,
                prompt_attention_mask=prompt_inputs.attention_mask,
            )

        # Synchronize CUDA before stopping timer
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed_time = time.perf_counter() - start_time

        # Convert to numpy array
        audio_arr = generation.cpu().numpy().squeeze()

        # Save audio file
        filename = f"{item['code']}_{lang.lower()}.wav"
        output_path = os.path.join(output_dir, filename)
        sf.write(output_path, audio_arr, sampling_rate)

        # Calculate audio duration
        duration = len(audio_arr) / sampling_rate
        rtf = elapsed_time / duration if duration > 0 else 0.0

        print(f"[{lang}]")
        print(f"  Saved to : {output_path}")
        print(f"  Duration : {duration:.2f}s")
        print(f"  Time     : {elapsed_time:.3f}s (RTF: {rtf:.2f}x)")
        print("-" * 50)



In [7]:

if __name__ == "__main__":
    generate_and_save_tts(samples)

Loading models on device: cpu...

Starting batch generation...
--------------------------------------------------
[Assamese]
  Saved to : tts_outputs\as_assamese.wav
  Duration : 5.32s
  Time     : 41.952s (RTF: 7.89x)
--------------------------------------------------
[Bodo]
  Saved to : tts_outputs\brx_bodo.wav
  Duration : 4.48s
  Time     : 33.985s (RTF: 7.58x)
--------------------------------------------------
[Nepali]
  Saved to : tts_outputs\ne_nepali.wav
  Duration : 4.95s
  Time     : 37.443s (RTF: 7.57x)
--------------------------------------------------
[Manipuri]
  Saved to : tts_outputs\mni_manipuri.wav
  Duration : 6.94s
  Time     : 54.355s (RTF: 7.83x)
--------------------------------------------------


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor
# recommended to run this on a gpu with flash_attn installed
# don't set attn_implemetation if you don't have flash_attn
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

src_lang, tgt_lang = "hin_Deva", "eng_Latn"
model_name = "ai4bharat/indictrans2-indic-en-dist-200M"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    trust_remote_code=True, 
    torch_dtype=torch.float16, # performance might slightly vary for bfloat16
    attn_implementation="flash_attention_2"
).to(DEVICE)

ip = IndicProcessor(inference=True)

input_sentences = [
    "जब मैं छोटा था, मैं हर रोज़ पार्क जाता था।",
    "हमने पिछले सप्ताह एक नई फिल्म देखी जो कि बहुत प्रेरणादायक थी।",
    "अगर तुम मुझे उस समय पास मिलते, तो हम बाहर खाना खाने चलते।",
    "मेरे मित्र ने मुझे उसके जन्मदिन की पार्टी में बुलाया है, और मैं उसे एक तोहफा दूंगा।",
]

batch = ip.preprocess_batch(
    input_sentences,
    src_lang=src_lang,
    tgt_lang=tgt_lang,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Tokenize the sentences and generate input encodings
inputs = tokenizer(
    batch,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    return_attention_mask=True,
).to(DEVICE)

# Generate translations using the model
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        use_cache=True,
        min_length=0,
        max_length=256,
        num_beams=5,
        num_return_sequences=1,
    )

# Decode the generated tokens into text
generated_tokens = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True,
)

# Postprocess the translations, including entity replacement
translations = ip.postprocess_batch(generated_tokens, lang=tgt_lang)

for input_sentence, translation in zip(input_sentences, translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")
